In [1]:
!where pip

C:\Users\Raunak\anaconda3\envs\py310\Scripts\pip.exe
C:\Program Files\Python312\Scripts\pip.exe
C:\Users\Raunak\anaconda3\Scripts\pip.exe


In [8]:
!WHERE python


C:\Users\Raunak\anaconda3\envs\py310\python.exe
C:\Program Files\Python312\python.exe
C:\Users\Raunak\AppData\Local\Microsoft\WindowsApps\python.exe
C:\Users\Raunak\anaconda3\python.exe


In [20]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- 1. CONFIGURATION ---

# ❗ 1. Update this to the path of your CSV file
INPUT_CSV = "Child Data - Form responses 1.csv" 

# ❗ 2. Update this to the path of the folder containing your downloaded images
IMAGE_DIR = "/"

# ❗ 3. This will be the name of the new file created
OUTPUT_CSV = "final_data_with_features.csv"


# --- Column names from your CSV ---
# Using the exact names you provided or from previous context
COL_NAME = "बच्चे का पूरा नाम"
COL_DOB = "बच्चे की जन्मतिथि"
COL_IMAGE_FILENAME = "आंगनवाड़ी केंद्र का कोड  " # This is now the image name

# These columns are needed for the model later, so we make sure to read them
# Using the exact names from your previous CSV dump
COL_HEIGHT = " ऊंचाई (सेंटीमीटर में,  उदाहरण: 89.5  )  "
COL_WEIGHT = "वजन (किलोग्राम में,  उदाहरण: 12.3  )  "

# --- 2. MEDIAPIPE SETUP (Unchanged) ---
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

def get_landmarks(image_path):
    """Extract pose landmarks from an image."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: Could not read image: {image_path}")
        return None

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)
    if not results.pose_landmarks:
        # print(f"Warning: No pose detected in {image_path}")
        return None
    landmarks = [(lm.x, lm.y) for lm in results.pose_landmarks.landmark]
    return np.array(landmarks)  # shape (33, 2)

# --- 3. FEATURE COMPUTATION (Unchanged) ---

IDX = {
    "NOSE":0, "L_SHO":11, "R_SHO":12, "L_ELB":13, "R_ELB":14, "L_WRI":15, "R_WRI":16,
    "L_HIP":23, "R_HIP":24, "L_KNE":25, "R_KNE":26, "L_ANK":27, "R_ANK":28,
    "L_HEEL":29, "R_HEEL":30, "L_FOOT":31, "R_FOOT":32
}

def _dist(lm, a, b): return float(np.linalg.norm(lm[a] - lm[b]))
def _mid(p, q): return (p + q) / 2.0

def _angle(a, b, c):
    v1, v2 = a - b, c - b
    num = float(np.dot(v1, v2))
    den = float(np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    cosang = np.clip(num / den, -1.0, 1.0)
    return float(np.arccos(cosang))  # radians

def compute_features(lm):
    # This entire function is identical to your original code
    L_SHO, R_SHO = lm[IDX["L_SHO"]], lm[IDX["R_SHO"]]
    L_HIP, R_HIP = lm[IDX["L_HIP"]], lm[IDX["R_HIP"]]
    L_ELB, R_ELB = lm[IDX["L_ELB"]], lm[IDX["R_ELB"]]
    L_WRI, R_WRI = lm[IDX["L_WRI"]], lm[IDX["R_WRI"]]
    L_KNE, R_KNE = lm[IDX["L_KNE"]], lm[IDX["R_KNE"]]
    L_ANK, R_ANK = lm[IDX["L_ANK"]], lm[IDX["R_ANK"]]
    L_HEEL, R_HEEL = lm[IDX["L_HEEL"]], lm[IDX["R_HEEL"]]
    L_FOOT, R_FOOT = lm[IDX["L_FOOT"]], lm[IDX["R_FOOT"]]
    NOSE = lm[IDX["NOSE"]]

    mid_sho = _mid(L_SHO, R_SHO)
    mid_hip = _mid(L_HIP, R_HIP)
    mid_ank = _mid(L_ANK, R_ANK)
    shoulder_width = _dist(lm, IDX["L_SHO"], IDX["R_SHO"])
    hip_width = _dist(lm, IDX["L_HIP"], IDX["R_HIP"])
    torsoL = _dist(lm, IDX["L_SHO"], IDX["L_HIP"])
    torsoR = _dist(lm, IDX["R_SHO"], IDX["R_HIP"])
    upperarmL = _dist(lm, IDX["L_SHO"], IDX["L_ELB"])
    forearmL = _dist(lm, IDX["L_ELB"], IDX["L_WRI"])
    thighL = _dist(lm, IDX["L_HIP"], IDX["L_KNE"])
    shankL = _dist(lm, IDX["L_KNE"], IDX["L_ANK"])
    legspan = float(np.linalg.norm(mid_hip - mid_ank)) + 1e-6
    headprox = float(np.linalg.norm(NOSE - mid_sho))
    armR = _dist(lm, IDX["R_SHO"], IDX["R_ELB"]) + _dist(lm, IDX["R_ELB"], IDX["R_WRI"])
    armL = upperarmL + forearmL
    ankle_span = _dist(lm, IDX["L_ANK"], IDX["R_ANK"])

    def nz(x): return x if x > 1e-6 else 1e-6

    feats = {}
    feats.update({
        "shoulder_width_px": shoulder_width, "hip_width_px": hip_width,
        "torso_len_L_px": torsoL, "torso_len_R_px": torsoR,
        "upperarm_L_px": upperarmL, "forearm_L_px": forearmL,
        "thigh_L_px": thighL, "shank_L_px": shankL,
    })
    feats.update({
        "hip_to_shoulder": hip_width / nz(shoulder_width),
        "shoulder_to_legspan": shoulder_width / nz(legspan),
        "torsoL_to_legspan": torsoL / nz(legspan),
        "torsoR_to_legspan": torsoR / nz(legspan),
        "torsoMean_to_legspan": (0.5 * (torsoL + torsoR)) / nz(legspan),
        "upperarmL_to_shoulder": upperarmL / nz(shoulder_width),
        "forearmL_to_shoulder": forearmL / nz(shoulder_width),
        "armL_to_legspan": armL / nz(legspan),
        "forearm_to_upperarm_L": forearmL / nz(upperarmL),
        "thighL_to_legspan": thighL / nz(legspan),
        "shankL_to_legspan": shankL / nz(legspan),
        "shank_to_thigh_L": shankL / nz(thighL),
        "legL_to_torsoMean": (thighL + shankL) / nz(0.5 * (torsoL + torsoR)),
        "ankle_span_to_shoulder": ankle_span / nz(shoulder_width),
        "headprox_to_shoulder": headprox / nz(shoulder_width),
        "headprox_to_legspan": headprox / nz(legspan),
        "shoulder_to_hip_midline": float(np.linalg.norm(mid_sho - mid_hip)) / nz(legspan),
        "shoulder_hip_ratio_LR": torsoL / nz(torsoR),
        "lateral_balance": (mid_sho[1] - mid_hip[1]) / nz(legspan),
        "foot_angle_L_normpi": _angle(L_ANK, L_HEEL, L_FOOT) / np.pi,
        "foot_angle_R_normpi": _angle(R_ANK, R_HEEL, R_FOOT) / np.pi,
        "arm_symmetry_L_over_R": armL / nz(armR),
    })
    return feats


# --- 4. MAIN PROCESSING LOOP ---

def main():
    # Load the main CSV
    try:
        labels_df = pd.read_csv(INPUT_CSV)
        print(labels_df.columns)
        print(f"Successfully loaded '{INPUT_CSV}'. Found {len(labels_df)} rows.")
    except FileNotFoundError:
        print(f"ERROR: Cannot find input CSV file at {INPUT_CSV}")
        print("Please update the 'INPUT_CSV' variable at the top of the script.")
        return
    except KeyError as e:
        print(f"ERROR: A required column is missing from the CSV: {e}")
        print("Please check the 'COL_...' variables at the top of the script.")
        return
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return
        
    # Check that the image directory exists
    if not os.path.isdir(IMAGE_DIR):
        print(f"ERROR: Image directory not found at {IMAGE_DIR}")
        print("Please update the 'IMAGE_DIR' variable at the top of the script.")
        return

    # This list will hold the feature dictionaries for each row
    all_features_data = []
    
    print(f"Starting feature extraction... Reading images from: {IMAGE_DIR}")
    
    # Loop through each row in the CSV
    for index, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Processing images"):
        try:
            # Get the filename from the specified column
            image_filename = str(row[COL_IMAGE_FILENAME])
            
            # Handle empty filenames
            if not image_filename or pd.isna(image_filename):
                print(f"Warning: Missing image filename at row {index}. Skipping.")
                all_features_data.append({}) # Append empty dict to keep rows aligned
                continue

            # Create the full path to the image
            img_path = os.path.join(IMAGE_DIR, image_filename)

            # Check if the image file actually exists
            if not os.path.exists(img_path):
                print(f"Warning: Image file not found at row {index}: {img_path}. Skipping.")
                all_features_data.append({})
                continue

            # 1. Get landmarks
            landmarks = get_landmarks(img_path)
            
            if landmarks is None:
                # print(f"Warning: No pose detected for {image_filename} (Row {index}). Skipping.")
                all_features_data.append({})
                continue  # skip if no pose detected

            # 2. Compute features
            features = compute_features(landmarks)
            all_features_data.append(features)

        except Exception as e:
            print(f"ERROR processing row {index} ({image_filename}): {e}. Skipping.")
            all_features_data.append({})

    print("\nFeature extraction complete.")
    
    # --- 5. APPEND FEATURES AND SAVE ---
    
    # Convert the list of feature dictionaries into a DataFrame
    # The index will align with labels_df, even if some rows are empty dicts
    features_df = pd.DataFrame(all_features_data, index=labels_df.index)
    
    # Get count of successful extractions
    successful_extractions = len(features_df.dropna(how='all'))
    print(f"Successfully extracted features for {successful_extractions} / {len(labels_df)} images.")

    # Concatenate the original DataFrame with the new features DataFrame
    # axis=1 means add as new columns (side-by-side)
    final_df = pd.concat([labels_df, features_df], axis=1)

    # Save the new combined DataFrame to a new CSV file
    try:
        final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print(f"✅ Success! All data with new features saved to: {OUTPUT_CSV}")
    except Exception as e:
        print(f"ERROR: Could not save final CSV file: {e}")


# Run the main processing function
if __name__ == "__main__":
    main()


Index(['Timestamp', 'बच्चे का पूरा नाम', 'बच्चे की जन्मतिथि  ',
       'Gender / लिंग', ' ऊंचाई (सेंटीमीटर में,  उदाहरण: 89.5  )  ',
       'वजन (किलोग्राम में,  उदाहरण: 12.3  )  ',
       'बच्चे की पूर्ण शरीर की तस्वीर (खड़े हुए स्थिति में)\n\n✳️ निर्देश (Instructions):\n\nबच्चा सीधे खड़ा हो, दोनों हाथ साइड में हों। \nसिर से पैर तक पूरा शरीर दिखाई दे। \nउचित दूरी (2 मीटर) से फोटो लें। \nतस्वीर में केवल एक ही बच्चा हो। \nभारी कपड़े ना पहनाए जाएँ। \nफोटो लेते समय मोबाइल को झुकाएँ नहीं, सीधा रखें।  \nफोटो का बैकग्राउंड सादा और साफ़ होना चाहिए (कोई डिज़ाइन या वस्तुएँ न हों)।  \nबच्चे के पीछे बस एक मैनुअल स्टैडियोमीटर रखें जिसकी ऊँचाई स्थायी रूप से 150 सेमी हो।\nहर फोटो में वही स्टैडियोमीटर (150 सेमी ऊँचाई वाला) उपयोग करें ताकि एक समान संदर्भ बना रहे।\nफोटो बच्चे की आँखों के स्तर से लें — बहुत ऊपर या नीचे से नहीं।',
       'आंगनवाड़ी केंद्र का नाम  ', 'आंगनवाड़ी केंद्र का कोड  ', 'Column 8'],
      dtype='object')
Successfully loaded 'Child Data - Form responses 1.csv'. Found 2467 rows.
St

Processing images:  17%|██████████                                                | 430/2467 [00:00<00:00, 4143.71it/s]

Processing images:  57%|████████████████████████████████▎                        | 1397/2467 [00:00<00:00, 4458.95it/s]

Processing images: 100%|█████████████████████████████████████████████████████████| 2467/2467 [00:00<00:00, 4089.28it/s]


Feature extraction complete.
Successfully extracted features for 0 / 2467 images.
✅ Success! All data with new features saved to: final_data_with_features.csv


In [4]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- 1. CONFIGURATION ---

# ❗ 1. Update this to the path of your CSV file
INPUT_CSV = "Child Data - Form responses 1.csv" 

# ❗ 2. Update this to the path of the folder containing your downloaded images
IMAGE_DIR = "data"

# ❗ 3. This will be the name of the new file created
OUTPUT_CSV = "final_data_with_features_2.csv"


# --- Column names from your CSV ---
COL_IMAGE_FILENAME = "आंगनवाड़ी केंद्र का कोड  " # The key to match on
COL_HEIGHT = " ऊंचाई (सेंटीमीटर में,  उदाहरण: 89.5  )  "
COL_WEIGHT = "वजन (किलोग्राम में,  उदाहरण: 12.3  )  "

# --- 2. MEDIAPIPE SETUP (Unchanged) ---
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

def get_landmarks(image_path):
    """Extract pose landmarks from an image."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: Could not read image: {image_path}")
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)
    if not results.pose_landmarks:
        return None
    landmarks = [(lm.x, lm.y) for lm in results.pose_landmarks.landmark]
    return np.array(landmarks)  # shape (33, 2)

# --- 3. FEATURE COMPUTATION (Unchanged) ---

IDX = {
    "NOSE":0, "L_SHO":11, "R_SHO":12, "L_ELB":13, "R_ELB":14, "L_WRI":15, "R_WRI":16,
    "L_HIP":23, "R_HIP":24, "L_KNE":25, "R_KNE":26, "L_ANK":27, "R_ANK":28,
    "L_HEEL":29, "R_HEEL":30, "L_FOOT":31, "R_FOOT":32
}

def _dist(lm, a, b): return float(np.linalg.norm(lm[a] - lm[b]))
def _mid(p, q): return (p + q) / 2.0
def _angle(a, b, c):
    v1, v2 = a - b, c - b
    num = float(np.dot(v1, v2))
    den = float(np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    cosang = np.clip(num / den, -1.0, 1.0)
    return float(np.arccos(cosang))  # radians

def compute_features(lm):
    # This entire function is identical to your original code
    L_SHO, R_SHO = lm[IDX["L_SHO"]], lm[IDX["R_SHO"]]
    L_HIP, R_HIP = lm[IDX["L_HIP"]], lm[IDX["R_HIP"]]
    L_ELB, R_ELB = lm[IDX["L_ELB"]], lm[IDX["R_ELB"]]
    L_WRI, R_WRI = lm[IDX["L_WRI"]], lm[IDX["R_WRI"]]
    L_KNE, R_KNE = lm[IDX["L_KNE"]], lm[IDX["R_KNE"]]
    L_ANK, R_ANK = lm[IDX["L_ANK"]], lm[IDX["R_ANK"]]
    L_HEEL, R_HEEL = lm[IDX["L_HEEL"]], lm[IDX["R_HEEL"]]
    L_FOOT, R_FOOT = lm[IDX["L_FOOT"]], lm[IDX["R_FOOT"]]
    NOSE = lm[IDX["NOSE"]]

    mid_sho = _mid(L_SHO, R_SHO); mid_hip = _mid(L_HIP, R_HIP); mid_ank = _mid(L_ANK, R_ANK)
    shoulder_width = _dist(lm, IDX["L_SHO"], IDX["R_SHO"]); hip_width = _dist(lm, IDX["L_HIP"], IDX["R_HIP"])
    torsoL = _dist(lm, IDX["L_SHO"], IDX["L_HIP"]); torsoR = _dist(lm, IDX["R_SHO"], IDX["R_HIP"])
    upperarmL = _dist(lm, IDX["L_SHO"], IDX["L_ELB"]); forearmL = _dist(lm, IDX["L_ELB"], IDX["L_WRI"])
    thighL = _dist(lm, IDX["L_HIP"], IDX["L_KNE"]); shankL = _dist(lm, IDX["L_KNE"], IDX["L_ANK"])
    legspan = float(np.linalg.norm(mid_hip - mid_ank)) + 1e-6
    headprox = float(np.linalg.norm(NOSE - mid_sho))
    armR = _dist(lm, IDX["R_SHO"], IDX["R_ELB"]) + _dist(lm, IDX["R_ELB"], IDX["R_WRI"])
    armL = upperarmL + forearmL; ankle_span = _dist(lm, IDX["L_ANK"], IDX["R_ANK"])

    def nz(x): return x if x > 1e-6 else 1e-6

    feats = {}; feats.update({
        "shoulder_width_px": shoulder_width, "hip_width_px": hip_width,
        "torso_len_L_px": torsoL, "torso_len_R_px": torsoR,
        "upperarm_L_px": upperarmL, "forearm_L_px": forearmL,
        "thigh_L_px": thighL, "shank_L_px": shankL,
    }); feats.update({
        "hip_to_shoulder": hip_width / nz(shoulder_width),
        "shoulder_to_legspan": shoulder_width / nz(legspan),
        "torsoL_to_legspan": torsoL / nz(legspan), "torsoR_to_legspan": torsoR / nz(legspan),
        "torsoMean_to_legspan": (0.5 * (torsoL + torsoR)) / nz(legspan),
        "upperarmL_to_shoulder": upperarmL / nz(shoulder_width),
        "forearmL_to_shoulder": forearmL / nz(shoulder_width),
        "armL_to_legspan": armL / nz(legspan), "forearm_to_upperarm_L": forearmL / nz(upperarmL),
        "thighL_to_legspan": thighL / nz(legspan), "shankL_to_legspan": shankL / nz(legspan),
        "shank_to_thigh_L": shankL / nz(thighL),
        "legL_to_torsoMean": (thighL + shankL) / nz(0.5 * (torsoL + torsoR)),
        "ankle_span_to_shoulder": ankle_span / nz(shoulder_width),
        "headprox_to_shoulder": headprox / nz(shoulder_width),
        "headprox_to_legspan": headprox / nz(legspan),
        "shoulder_to_hip_midline": float(np.linalg.norm(mid_sho - mid_hip)) / nz(legspan),
        "shoulder_hip_ratio_LR": torsoL / nz(torsoR),
        "lateral_balance": (mid_sho[1] - mid_hip[1]) / nz(legspan),
        "foot_angle_L_normpi": _angle(L_ANK, L_HEEL, L_FOOT) / np.pi,
        "foot_angle_R_normpi": _angle(R_ANK, R_HEEL, R_FOOT) / np.pi,
        "arm_symmetry_L_over_R": armL / nz(armR),
    }); return feats


# --- 4. MAIN PROCESSING LOOP (NEW LOGIC) ---

def main():
    # --- 1. Load CSV (Lookup Table) ---
    try:
        labels_df = pd.read_csv(INPUT_CSV)
        # IMPORTANT: Ensure the matching column is treated as a string
        labels_df[COL_IMAGE_FILENAME] = labels_df[COL_IMAGE_FILENAME].astype(str)
        print(f"Successfully loaded '{INPUT_CSV}'. Found {len(labels_df)} rows.")
    except FileNotFoundError:
        print(f"ERROR: Cannot find input CSV file at {INPUT_CSV}")
        return
    except KeyError as e:
        print(f"ERROR: A required column is missing from the CSV: {e}")
        return
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # --- 2. Get Image Files ---
    try:
        # Get all files in the directory
        all_files = os.listdir(IMAGE_DIR)
        # Filter for common image extensions
        image_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"ERROR: No image files (.png, .jpg, .jpeg) found in {IMAGE_DIR}")
            return
        print(f"Found {len(image_files)} images in '{IMAGE_DIR}'.")
    except FileNotFoundError:
        print(f"ERROR: Image directory not found at {IMAGE_DIR}")
        return
    except Exception as e:
        print(f"Error reading image directory: {e}")
        return

    # --- 3. Create a fast lookup set from the CSV data ---
    # This is much faster than searching the DataFrame every time
    csv_filenames_lookup = set(labels_df[COL_IMAGE_FILENAME].values)

    # This list will store the combined data (original row + new features)
    all_processed_data = []

    print("Starting feature extraction (Image-first matching)...")
    
    # --- 4. Loop through IMAGE FILES first ---
    for image_filename in tqdm(image_files, desc="Processing images"):
        try:
            # --- 5. Check if this image has a match in the CSV ---
            if image_filename not in csv_filenames_lookup:
                # print(f"Info: Image {image_filename} not found in CSV. Skipping.")
                continue

            # --- 6. MATCH FOUND: Process the image ---
            
            # Get the full path to the image
            img_path = os.path.join(IMAGE_DIR, image_filename)
            
            # Get the matching row from the CSV
            # We use .iloc[0] because we expect only one match
            matched_row = labels_df[labels_df[COL_IMAGE_FILENAME] == image_filename].iloc[0]
            
            # Convert the original row to a dictionary
            combined_data = matched_row.to_dict()

            # --- 7. Run Feature Extraction ---
            landmarks = get_landmarks(img_path)
            
            if landmarks is None:
                # We found the image, but pose detection failed.
                # We still save the original data.
                # print(f"Warning: No pose detected for {image_filename}.")
                all_processed_data.append(combined_data) # Add original data
                continue

            features = compute_features(landmarks)
            
            # --- 8. Merge and Append ---
            # Add the new features to the dictionary
            combined_data.update(features)
            
            # Add the complete record to our list
            all_processed_data.append(combined_data)

        except Exception as e:
            print(f"ERROR processing {image_filename}: {e}. Skipping.")

    # --- 9. Save Final Data ---
    if not all_processed_data:
        print("\nNo matching images were found or processed. No output file created.")
        return

    # Convert the list of dictionaries into the final DataFrame
    final_df = pd.DataFrame(all_processed_data)
    
    print(f"\nSuccessfully processed {len(final_df)} matching images.")
    
    try:
        final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print(f"✅ Success! All matched data with new features saved to: {OUTPUT_CSV}")
    except Exception as e:
        print(f"ERROR: Could not save final CSV file: {e}")


# Run the main processing function
if __name__ == "__main__":
    main()


Successfully loaded 'Child Data - Form responses 1.csv'. Found 2467 rows.
Found 300 images in 'data'.
Starting feature extraction (Image-first matching)...


Processing images: 100%|█████████████████████████████████████████████████████████████| 300/300 [01:31<00:00,  3.29it/s]


Successfully processed 275 matching images.
✅ Success! All matched data with new features saved to: final_data_with_features_2.csv
